<a href="https://colab.research.google.com/github/kasrasa/ViT-VLM-experiments/blob/VLM-Experiments/VLM_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch
!pip install -q timm
!pip install -q evaluate
!pip install -q peft
!pip install --upgrade -q torchao
!pip install -q scikit-learn
!pip install -q matplotlib seaborn accelerate

In [ ]:
import os
import re
import gc
import time
import random
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm
from datasets import load_dataset, DatasetDict

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from torchvision.transforms import (
    RandomResizedCrop,
    Resize,
    CenterCrop,
    Compose,
    Normalize,
    ToTensor,
)

from transformers import (
    DefaultDataCollator,
    AutoImageProcessor,
    AutoModelForImageClassification,
    AutoProcessor,
    AutoModelForImageTextToText,
    TrainingArguments,
    Trainer,
)

In [ ]:
# Set to True to always fine-tune, False to load existing checkpoints if available
FORCE_FINE_TUNING = True

In [ ]:
data_collator = DefaultDataCollator()

In [ ]:
def get_image_size(image_processor):
    if "shortest_edge" in image_processor.size:
        return image_processor.size["shortest_edge"]
    return image_processor.size["height"]

def apply_transforms(examples, image_processor, is_train=True):
    image_size = get_image_size(image_processor)
    normalize = Normalize(
        mean=image_processor.image_mean,
        std=image_processor.image_std,
    )

    if is_train:
        transform = Compose([
            RandomResizedCrop(image_size),
            ToTensor(),
            normalize,
        ])
    else:
        transform = Compose([
            Resize(image_size),
            CenterCrop(image_size),
            ToTensor(),
            normalize,
        ])

    examples["pixel_values"] = [
        transform(img.convert("RGB")) for img in examples["image"]
    ]
    del examples["image"]
    return examples

In [ ]:
# -----------------------------------------------------
# Shared classification metric helpers
# -----------------------------------------------------
# Used by:
#   1. ConvNeXt logits
#   2. SmolVLM forced-choice logits
#   3. SmolVLM single-output predictions, partially


def softmax_np(logits):
    """
    Numerically stable softmax for [N, C] logits.
    """
    logits = np.asarray(logits, dtype=np.float64)
    logits = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def expected_calibration_error(probs, labels, n_bins=10):
    """
    Expected Calibration Error.

    Compares confidence with actual accuracy in confidence bins.
    """
    probs = np.asarray(probs)
    labels = np.asarray(labels).astype(int)

    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    correct = predictions == labels

    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lower = bin_edges[i]
        upper = bin_edges[i + 1]

        if i == 0:
            in_bin = (confidences >= lower) & (confidences <= upper)
        else:
            in_bin = (confidences > lower) & (confidences <= upper)

        prop_in_bin = np.mean(in_bin)

        if prop_in_bin > 0:
            avg_confidence = np.mean(confidences[in_bin])
            avg_accuracy = np.mean(correct[in_bin])
            ece += prop_in_bin * abs(avg_confidence - avg_accuracy)

    return float(ece)


def top_k_accuracy_from_probs(probs, labels, k=5):
    """
    Computes top-k accuracy from class probabilities.
    """
    probs = np.asarray(probs)
    labels = np.asarray(labels).astype(int)

    k = min(k, probs.shape[1])

    top_k_predictions = np.argpartition(probs, -k, axis=1)[:, -k:]
    top_k_correct = np.any(top_k_predictions == labels[:, None], axis=1)

    return float(np.mean(top_k_correct))


def compute_logits_metrics(logits, labels, n_bins=10):
    """
    Computes all probability-based classification metrics.

    Input:
        logits: [N, C]
        labels: [N]

    Used for:
        - ConvNeXt
        - SmolVLM forced-choice class scores
    """
    logits = np.asarray(logits)
    labels = np.asarray(labels).astype(int)

    probs = softmax_np(logits)
    predictions = np.argmax(probs, axis=1)

    num_classes = probs.shape[1]

    top1_confidence = np.max(probs, axis=1)

    if num_classes > 1:
        top2_confidence = np.partition(probs, -2, axis=1)[:, -2]
    else:
        top2_confidence = np.zeros_like(top1_confidence)

    top1_top2_margin = top1_confidence - top2_confidence

    correct_mask = predictions == labels
    wrong_mask = ~correct_mask

    correct_mean_confidence = (
        float(np.mean(top1_confidence[correct_mask]))
        if np.any(correct_mask)
        else 0.0
    )

    wrong_mean_confidence = (
        float(np.mean(top1_confidence[wrong_mask]))
        if np.any(wrong_mask)
        else 0.0
    )

    eps = 1e-12
    true_class_probs = probs[np.arange(len(labels)), labels]
    nll = -np.mean(np.log(np.clip(true_class_probs, eps, 1.0)))

    one_hot = np.zeros_like(probs)
    one_hot[np.arange(len(labels)), labels] = 1.0
    brier_score = np.mean(np.sum((probs - one_hot) ** 2, axis=1))

    weighted_f1 = f1_score(
        labels,
        predictions,
        average="weighted",
        zero_division=0,
    )

    metrics = {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro", zero_division=0),
        "weighted_f1": weighted_f1,

        # Compatibility alias for older summary tables.
        "f1": weighted_f1,

        "top_5_accuracy": top_k_accuracy_from_probs(probs, labels, k=5),
        "mean_top1_confidence": float(np.mean(top1_confidence)),
        "mean_top1_top2_margin": float(np.mean(top1_top2_margin)),
        "correct_mean_confidence": correct_mean_confidence,
        "wrong_mean_confidence": wrong_mean_confidence,
        "ece": expected_calibration_error(probs, labels, n_bins=n_bins),
        "nll": float(nll),
        "brier_score": float(brier_score),
    }

    return metrics, probs, predictions


def compute_prediction_metrics(
    predictions,
    labels,
    class_ids,
    invalid_id=None,
):
    """
    Computes simple classification metrics from predicted class IDs.

    Used for:
        - SmolVLM single-output generation

    This does not compute probability metrics because single-output VLM
    generation does not produce class probabilities.
    """
    predictions = np.asarray(predictions).astype(int)
    labels = np.asarray(labels).astype(int)

    if invalid_id is not None:
        predictions_for_metrics = np.where(predictions < 0, invalid_id, predictions)
        invalid_rate = float(np.mean(predictions < 0))
    else:
        predictions_for_metrics = predictions
        invalid_rate = 0.0

    accuracy = accuracy_score(labels, predictions_for_metrics)

    macro_f1 = f1_score(
        labels,
        predictions_for_metrics,
        labels=class_ids,
        average="macro",
        zero_division=0,
    )

    weighted_f1 = f1_score(
        labels,
        predictions_for_metrics,
        labels=class_ids,
        average="weighted",
        zero_division=0,
    )

    return {
        "accuracy": accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "invalid_output_rate": invalid_rate,
    }


def make_classification_report_df(
    labels,
    predictions,
    class_ids,
    target_names,
):
    """
    Builds a sklearn classification report as a DataFrame.
    """
    report = classification_report(
        labels,
        predictions,
        labels=class_ids,
        target_names=target_names,
        output_dict=True,
        zero_division=0,
    )

    report_df = pd.DataFrame(report).transpose()

    return report_df


def plot_confusion_matrix(
    labels,
    predictions,
    class_ids,
    target_names,
    model_display_name,
    dataset_name,
    normalize_cm=False,
    figsize=(12, 10),
):
    """
    Shared confusion matrix plotting function.
    """
    cm = confusion_matrix(
        labels,
        predictions,
        labels=class_ids,
    )

    if normalize_cm:
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_to_plot = np.divide(
            cm.astype("float"),
            row_sums,
            out=np.zeros_like(cm, dtype=float),
            where=row_sums != 0,
        )
        fmt = ".2f"
    else:
        cm_to_plot = cm
        fmt = "g"

    plt.figure(figsize=figsize)

    if len(class_ids) > 21:
        sns.heatmap(
            cm_to_plot,
            cmap="Blues",
            xticklabels=False,
            yticklabels=False,
            cbar=True,
        )
        plt.title(
            f"Confusion Matrix for {model_display_name} on {dataset_name}\n"
            f"Labels omitted because there are {len(class_ids)} labels"
        )
    else:
        sns.heatmap(
            cm_to_plot,
            annot=True,
            fmt=fmt,
            cmap="Blues",
            xticklabels=target_names,
            yticklabels=target_names,
        )
        plt.title(f"Confusion Matrix for {model_display_name} on {dataset_name}")

    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.tight_layout()
    plt.show()

    return cm


def display_strongest_and_weakest_classes(report_df, target_names, top_n=10):
    """
    Displays weakest and strongest classes by F1-score.
    """
    class_metrics_df = report_df.loc[target_names].copy()
    class_metrics_df["f1-score"] = pd.to_numeric(
        class_metrics_df["f1-score"],
        errors="coerce",
    )

    print(f"\nTop {top_n} weakest classes by F1-score:")
    weakest_classes = class_metrics_df.sort_values(
        "f1-score",
        ascending=True,
    ).head(top_n)

    display(weakest_classes[["precision", "recall", "f1-score", "support"]].round(4))

    print(f"\nTop {top_n} strongest classes by F1-score:")
    strongest_classes = class_metrics_df.sort_values(
        "f1-score",
        ascending=False,
    ).head(top_n)

    display(strongest_classes[["precision", "recall", "f1-score", "support"]].round(4))


def compute_metrics(eval_pred):
    """
    Hugging Face Trainer-compatible metric function.

    This is now a thin wrapper around compute_logits_metrics(...).
    """
    logits = eval_pred.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    labels = eval_pred.label_ids

    metrics, _, _ = compute_logits_metrics(logits, labels)

    return metrics

In [ ]:
# -----------------------------------------------------
# Centralized logits-based evaluation
# -----------------------------------------------------
# Used for:
#   1. ConvNeXt direct classifier logits
#   2. SmolVLM forced-choice class scores
#
# Input:
#   logits: [N, C]
#   labels: [N]

class ClassificationLogitsOutput:
    """
    Minimal prediction container compatible with compute_metrics().
    """
    def __init__(self, predictions, label_ids, metrics=None):
        self.predictions = predictions
        self.label_ids = label_ids
        self.metrics = metrics or {}


def get_trainer_logits_and_labels(trainer, dataset):
    """
    Extracts class logits and labels from a Hugging Face Trainer model.
    """
    predictions_output = trainer.predict(dataset)

    logits = predictions_output.predictions

    if isinstance(logits, tuple):
        logits = logits[0]

    labels = predictions_output.label_ids
    eval_loss = predictions_output.metrics.get("test_loss", None)

    return np.asarray(logits), np.asarray(labels).astype(int), eval_loss


def evaluate_logits_and_plot(
    logits,
    labels,
    id2label_mapping,
    dataset_name,
    num_labels,
    model_display_name,
    training_time=None,
    eval_loss=None,
    normalize_cm=False,
    extra_prediction_columns=None,
):
    """
    Centralized evaluation for models that produce class logits.

    Valid for:
        - ConvNeXt
        - forced-choice VLM scoring

    Not valid for:
        - single-output VLM generation
    """
    print(f"\n--- Evaluating {model_display_name} on {dataset_name} ---")

    logits = np.asarray(logits)
    labels = np.asarray(labels).astype(int)

    assert logits.ndim == 2, f"Expected logits with shape [N, C], got {logits.shape}"
    assert labels.ndim == 1, f"Expected labels with shape [N], got {labels.shape}"
    assert logits.shape[0] == labels.shape[0], "Number of logits and labels does not match."
    assert logits.shape[1] == num_labels, f"Expected {num_labels} classes, got {logits.shape[1]}"

    metrics, probs, predicted_labels = compute_logits_metrics(logits, labels)

    if eval_loss is not None:
        metrics["eval_loss"] = float(eval_loss)

    print("\nCore metrics:")
    print(f"Accuracy:                {metrics['accuracy']:.4f}")
    print(f"Macro F1:                {metrics['macro_f1']:.4f}")
    print(f"Weighted F1:             {metrics['weighted_f1']:.4f}")
    print(f"Top-5 Accuracy:          {metrics['top_5_accuracy']:.4f}")

    if eval_loss is not None:
        print(f"Eval Loss:               {metrics['eval_loss']:.4f}")

    print("\nConfidence / calibration metrics:")
    print(f"Mean Top-1 Confidence:   {metrics['mean_top1_confidence']:.4f}")
    print(f"Mean Top1-Top2 Margin:   {metrics['mean_top1_top2_margin']:.4f}")
    print(f"Correct Mean Confidence: {metrics['correct_mean_confidence']:.4f}")
    print(f"Wrong Mean Confidence:   {metrics['wrong_mean_confidence']:.4f}")
    print(f"ECE:                     {metrics['ece']:.4f}")
    print(f"NLL:                     {metrics['nll']:.4f}")
    print(f"Brier Score:             {metrics['brier_score']:.4f}")

    if training_time is not None:
        print(f"\nTraining time for {model_display_name}: {training_time:.2f} seconds")

    print("\nLabel sanity check:")
    print(f"Number of samples:                 {len(labels)}")
    print(f"Number of unique true labels:       {len(np.unique(labels))}")
    print(f"Number of unique predicted labels:  {len(np.unique(predicted_labels))}")

    top1_confidence = np.max(probs, axis=1)

    if probs.shape[1] > 1:
        top2_confidence = np.partition(probs, -2, axis=1)[:, -2]
    else:
        top2_confidence = np.zeros_like(top1_confidence)

    margins = top1_confidence - top2_confidence
    correct = predicted_labels == labels

    confidence_df = pd.DataFrame({
        "true_label_id": labels,
        "pred_label_id": predicted_labels,
        "true_label": [id2label_mapping[int(i)] for i in labels],
        "pred_label": [id2label_mapping[int(i)] for i in predicted_labels],
        "top1_confidence": top1_confidence,
        "top2_confidence": top2_confidence,
        "top1_top2_margin": margins,
        "correct": correct,
    })

    if extra_prediction_columns is not None:
        for col_name, values in extra_prediction_columns.items():
            confidence_df[col_name] = values

    print("\nConfidence summary:")
    display(
        confidence_df.groupby("correct")[[
            "top1_confidence",
            "top1_top2_margin",
        ]]
        .agg(["mean", "median", "min", "max", "count"])
        .round(4)
    )

    print("\nMost confident wrong predictions:")
    wrong_predictions = confidence_df[confidence_df["correct"] == False]

    if len(wrong_predictions) > 0:
        display(
            wrong_predictions
            .sort_values("top1_confidence", ascending=False)
            .head(10)
            .round(4)
        )
    else:
        print("No wrong predictions found.")

    print("\nLowest-margin predictions:")
    display(
        confidence_df
        .sort_values("top1_top2_margin", ascending=True)
        .head(10)
        .round(4)
    )

    class_ids = list(range(num_labels))
    target_names = [id2label_mapping[i] for i in class_ids]

    plot_confusion_matrix(
        labels=labels,
        predictions=predicted_labels,
        class_ids=class_ids,
        target_names=target_names,
        model_display_name=model_display_name,
        dataset_name=dataset_name,
        normalize_cm=normalize_cm,
        figsize=(12, 10),
    )

    print(f"\n--- Per-class metrics for {model_display_name} ---")

    report_df = make_classification_report_df(
        labels=labels,
        predictions=predicted_labels,
        class_ids=class_ids,
        target_names=target_names,
    )

    display_strongest_and_weakest_classes(
        report_df=report_df,
        target_names=target_names,
        top_n=10,
    )

    metrics["classification_report_df"] = report_df
    metrics["confidence_df"] = confidence_df
    metrics["logits"] = logits
    metrics["labels"] = labels
    metrics["probs"] = probs
    metrics["predictions"] = predicted_labels

    return metrics

In [ ]:
# -----------------------------------------------------
# Centralized prediction-based evaluation
# -----------------------------------------------------
# Used for:
#   1. SmolVLM single-output generation
#   2. Any model that only returns predicted class IDs
#
# Input:
#   predictions: [N]
#   labels: [N]
#
# Invalid generated VLM outputs should be represented as -1.

def evaluate_predictions_and_plot(
    predictions,
    labels,
    id2label_mapping,
    dataset_name,
    num_labels,
    model_display_name,
    outputs_df=None,
    normalize_cm=False,
    include_invalid_output=True,
):
    """
    Evaluates models that output one predicted class ID per image.

    This does not compute:
        - confidence
        - ECE
        - NLL
        - Brier score
        - top-5 accuracy

    because class probabilities are not available.
    """
    print(f"\n--- Evaluating {model_display_name} on {dataset_name} ---")

    predictions = np.asarray(predictions).astype(int)
    labels = np.asarray(labels).astype(int)

    assert predictions.ndim == 1, f"Expected predictions with shape [N], got {predictions.shape}"
    assert labels.ndim == 1, f"Expected labels with shape [N], got {labels.shape}"
    assert predictions.shape[0] == labels.shape[0], "Number of predictions and labels does not match."

    class_ids = list(range(num_labels))
    target_names = [id2label_mapping[i] for i in class_ids]

    has_invalid = np.any(predictions < 0)

    if include_invalid_output and has_invalid:
        invalid_id = num_labels
        predictions_for_metrics = np.where(predictions < 0, invalid_id, predictions)
        cm_class_ids = class_ids + [invalid_id]
        cm_target_names = target_names + ["INVALID_OUTPUT"]
    else:
        invalid_id = None
        predictions_for_metrics = predictions
        cm_class_ids = class_ids
        cm_target_names = target_names

    metrics = compute_prediction_metrics(
        predictions=predictions,
        labels=labels,
        class_ids=class_ids,
        invalid_id=invalid_id,
    )

    print("\nCore metrics:")
    print(f"Accuracy:                {metrics['accuracy']:.4f}")
    print(f"Macro F1:                {metrics['macro_f1']:.4f}")
    print(f"Weighted F1:             {metrics['weighted_f1']:.4f}")

    if include_invalid_output:
        print(f"Invalid output rate:     {metrics['invalid_output_rate']:.4f}")

    valid_predictions = predictions[predictions >= 0]

    print("\nLabel sanity check:")
    print(f"Number of samples:                 {len(labels)}")
    print(f"Number of unique true labels:       {len(np.unique(labels))}")
    print(f"Number of unique predicted labels:  {len(np.unique(valid_predictions))}")
    print(f"Number of invalid outputs:          {int(np.sum(predictions < 0))}")

    if outputs_df is not None:
        output_cols = [
            "sample_index",
            "true_label",
            "pred_label",
            "valid_output",
            "correct",
            "raw_output",
        ]

        existing_cols = [col for col in output_cols if col in outputs_df.columns]

        print("\nSample generated outputs:")
        display(outputs_df[existing_cols].head(20))

        wrong_df = outputs_df[outputs_df["correct"] == False].copy()

        print("\nSample wrong or invalid predictions:")
        if len(wrong_df) > 0:
            display(wrong_df[existing_cols].head(20))
        else:
            print("No wrong predictions found.")

    plot_confusion_matrix(
        labels=labels,
        predictions=predictions_for_metrics,
        class_ids=cm_class_ids,
        target_names=cm_target_names,
        model_display_name=model_display_name,
        dataset_name=dataset_name,
        normalize_cm=normalize_cm,
        figsize=(13, 10) if include_invalid_output and has_invalid else (12, 10),
    )

    print(f"\n--- Per-class metrics for {model_display_name} ---")

    report_df = make_classification_report_df(
        labels=labels,
        predictions=predictions_for_metrics,
        class_ids=class_ids,
        target_names=target_names,
    )

    display_strongest_and_weakest_classes(
        report_df=report_df,
        target_names=target_names,
        top_n=10,
    )

    metrics["classification_report_df"] = report_df
    metrics["predictions"] = predictions
    metrics["labels"] = labels
    metrics["outputs_df"] = outputs_df

    return metrics

In [ ]:
# -----------------------------
# Balanced Food101 subset config
# -----------------------------
DATASET_NAME = "ethz/food101"
NUM_SELECTED_CLASSES = 20
SAMPLES_PER_CLASS = 250
TEST_SIZE = 0.20
SEED = 42

# Load the full Food101 training split.
# We use the train split and create our own stratified train/validation split
# because this experiment is meant to compare fine-tuning strategies cheaply.
food_full_train = load_dataset(DATASET_NAME, split="train")
original_labels = food_full_train.features["label"].names

rng = random.Random(SEED)

# Group dataset indices by original Food101 class id
label_to_indices = defaultdict(list)
for idx, label_id in enumerate(food_full_train["label"]):
    label_to_indices[int(label_id)].append(idx)

# Keep only classes that have enough examples
eligible_label_ids = [
    label_id
    for label_id, indices in label_to_indices.items()
    if len(indices) >= SAMPLES_PER_CLASS
]

if len(eligible_label_ids) < NUM_SELECTED_CLASSES:
    raise ValueError(
        f"Only {len(eligible_label_ids)} classes have at least "
        f"{SAMPLES_PER_CLASS} samples. Need {NUM_SELECTED_CLASSES}."
    )

# Select 20 classes reproducibly
selected_old_label_ids = sorted(rng.sample(eligible_label_ids, NUM_SELECTED_CLASSES))

# Select exactly 250 images per selected class
selected_indices = []
for old_label_id in selected_old_label_ids:
    indices = label_to_indices[old_label_id].copy()
    rng.shuffle(indices)
    selected_indices.extend(indices[:SAMPLES_PER_CLASS])

rng.shuffle(selected_indices)

food_subset = food_full_train.select(selected_indices)

print(f"Total selected images: {len(food_subset)}")
print(f"Selected classes: {len(selected_old_label_ids)}")
print("Selected class names:")
for old_label_id in selected_old_label_ids:
    print(f"  old_id={old_label_id:3d} -> {original_labels[old_label_id]}")

# Stratified split while labels are still original Food101 label IDs
food = food_subset.train_test_split(
    test_size=TEST_SIZE,
    shuffle=True,
    seed=SEED,
    stratify_by_column="label",
)

# Remap selected original labels to contiguous labels 0..19.
# This is important because the classifier head will have exactly 20 outputs.
old_to_new = {
    old_label_id: new_label_id
    for new_label_id, old_label_id in enumerate(selected_old_label_ids)
}

new_to_old = {
    new_label_id: old_label_id
    for old_label_id, new_label_id in old_to_new.items()
}

labels = [
    original_labels[new_to_old[new_label_id]]
    for new_label_id in range(NUM_SELECTED_CLASSES)
]

id2label = {
    new_label_id: label_name
    for new_label_id, label_name in enumerate(labels)
}

label2id = {
    label_name: new_label_id
    for new_label_id, label_name in id2label.items()
}

def remap_label(example):
    example["label"] = old_to_new[int(example["label"])]
    return example

food = DatasetDict({
    "train": food["train"].map(remap_label),
    "test": food["test"].map(remap_label),
})

# Sanity checks
train_counts = Counter(food["train"]["label"])
test_counts = Counter(food["test"]["label"])

print("\nAfter remapping:")
print(f"Train size: {len(food['train'])}")
print(f"Validation size: {len(food['test'])}")
print(f"Number of labels: {len(labels)}")
print(f"Train class counts: {sorted(train_counts.items())}")
print(f"Validation class counts: {sorted(test_counts.items())}")

assert len(food["train"]) + len(food["test"]) == NUM_SELECTED_CLASSES * SAMPLES_PER_CLASS
assert set(train_counts.keys()) == set(range(NUM_SELECTED_CLASSES))
assert set(test_counts.keys()) == set(range(NUM_SELECTED_CLASSES))

In [ ]:
print(f"Number of selected labels: {len(labels)}")
print("id2label:")
for class_id, class_name in id2label.items():
    print(f"  {class_id}: {class_name}")

print("\nlabel2id:")
for class_name, class_id in label2id.items():
    print(f"  {class_name}: {class_id}")


In [ ]:
def conditional_train_model(model, trainer, training_args, model_name):
    output_dir = training_args.output_dir
    last_checkpoint = None
    if os.path.isdir(output_dir) and not FORCE_FINE_TUNING:
        try:
            last_checkpoint = get_last_checkpoint(output_dir)
            print(f"Loading checkpoint for {model_name} from {last_checkpoint}")
            model = AutoModelForImageClassification.from_pretrained(last_checkpoint)
            # Ensure model is on the correct device if not already handled by from_pretrained
            if torch.cuda.is_available():
                model.to('cuda')
        except Exception as e:
            print(f"Could not load checkpoint for {model_name}: {e}. Retraining.")
            last_checkpoint = None

    if last_checkpoint is None or FORCE_FINE_TUNING:
        print(f"Fine-tuning {model_name}...")
        train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
        trainer.save_model()
        metrics = train_result.metrics
        trainer.log_metrics("train", metrics)
        trainer.save_metrics("train", metrics)
        trainer.save_state()
        return metrics.get('train_runtime', 0.0)
    else:
        print(f"Skipping fine-tuning for {model_name}, loaded from checkpoint.")
        # If not fine-tuning, we might still want to run an evaluation to get metrics
        # or just return 0 for train_runtime as no training occurred.
        return 0.0

# Helper for get_last_checkpoint, usually from transformers.trainer_utils
def get_last_checkpoint(checkpoint_dir):
    checkpoints = [path for path in os.listdir(checkpoint_dir) if path.startswith("checkpoint-")]
    if not checkpoints:
        return None
    return os.path.join(checkpoint_dir, max(checkpoints, key=lambda x: int(x.split('-')[-1])))

In [ ]:
# ConvNeXt-Tiny model from Hugging Face
convnext_model_id_hf = "facebook/convnext-tiny-224"

# Load the ConvNeXt-specific image processor
convnext_image_processor = AutoImageProcessor.from_pretrained(convnext_model_id_hf)

# Apply ConvNeXt-specific transforms to the already split balanced Food101 subset
food_convnext = food.copy()

food_convnext["train"] = food_convnext["train"].with_transform(
    lambda examples: apply_transforms(
        examples,
        convnext_image_processor,
        is_train=True
    )
)

food_convnext["test"] = food_convnext["test"].with_transform(
    lambda examples: apply_transforms(
        examples,
        convnext_image_processor,
        is_train=False
    )
)

In [ ]:
# Load the ConvNeXt-Tiny model with a new 20-class classification head
convnext_model_hf = AutoModelForImageClassification.from_pretrained(
    convnext_model_id_hf,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

# Ensure all parameters require gradients for full fine-tuning
for param in convnext_model_hf.parameters():
    param.requires_grad = True

print("ConvNeXt-Tiny model loaded and configured for full fine-tuning.")
print(convnext_model_hf)

# Sanity checks
print("Number of labels:", convnext_model_hf.config.num_labels)
print("id2label length:", len(convnext_model_hf.config.id2label))
print("label2id length:", len(convnext_model_hf.config.label2id))

In [ ]:
# Define TrainingArguments for ConvNeXt-Tiny full fine-tuning
training_args_convnext_hf = TrainingArguments(
    output_dir="/content/drive/MyDrive/ViTExperiments/food101_20cls_250_convnext_tiny_full_finetune",
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    warmup_steps=10,
    logging_steps=10,
    run_name="food101_convnext_tiny_full_finetune",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

# Initialize the Trainer for ConvNeXt-Tiny
trainer_convnext_hf = Trainer(
    model=convnext_model_hf,
    args=training_args_convnext_hf,
    data_collator=data_collator,
    train_dataset=food_convnext["train"],
    eval_dataset=food_convnext["test"],
    processing_class=convnext_image_processor,
    compute_metrics=compute_metrics,
)

print("Starting ConvNeXt-Tiny full model fine-tuning...")

convnext_hf_full_train_time = conditional_train_model(
    convnext_model_hf,
    trainer_convnext_hf,
    training_args_convnext_hf,
    "ConvNeXt-Tiny HF Full Fine-tune"
)

### VLM Model Loading and Inference Setup

In [ ]:
# SmolVLM model from Hugging Face.
# This is used as a zero-shot VLM classifier by scoring each candidate class label
# with image-conditioned log-likelihood, not by generating free-form text.
smolvlm_model_id = "HuggingFaceTB/SmolVLM-Instruct"

vlm_device = "cuda" if torch.cuda.is_available() else "cpu"
vlm_dtype = torch.float16 if vlm_device == "cuda" else torch.float32

processor_smolvlm = AutoProcessor.from_pretrained(smolvlm_model_id)
model_smolvlm = AutoModelForImageTextToText.from_pretrained(
    smolvlm_model_id,
    torch_dtype=vlm_dtype,
)
model_smolvlm.to(vlm_device)
model_smolvlm.eval()

print(f"SmolVLM processor and model loaded on {vlm_device} with dtype={vlm_dtype}.")

### VLM forced-choice class-logit scoring setup


In [ ]:
# -----------------------------------------------------
# Shared SmolVLM utilities
# -----------------------------------------------------
# Used by both:
#   1. Forced-choice scoring: image + prompt + candidate -> class scores
#   2. Single-output generation: image + prompt -> one generated class label


def vlm_label_name(label_name):
    """Convert dataset label style to natural language style for the VLM."""
    return str(label_name).replace("_", " ")


def normalize_label_text(text):
    """
    Normalize generated text and label names so outputs like:
        'Caesar salad.', 'caesar_salad', 'The answer is caesar salad'
    can all match the class label 'caesar_salad'.
    """
    text = str(text).lower().strip()
    text = text.replace("_", " ")
    text = text.replace("-", " ")
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    prefixes = [
        "the answer is",
        "answer is",
        "answer",
        "the class is",
        "class is",
        "class",
        "the label is",
        "label is",
        "label",
        "the food is",
        "food is",
        "food",
        "this is",
        "it is",
        "it looks like",
    ]

    for prefix in prefixes:
        if text.startswith(prefix + " "):
            text = text[len(prefix):].strip()

    return text


def build_vlm_label_maps(id2label_mapping):
    """
    Creates reusable label mappings.

    Example:
        id2label_mapping[12] = 'hot_and_sour_soup'

    Produces:
        id_to_dataset_label[12] = 'hot_and_sour_soup'
        id_to_natural_label[12] = 'hot and sour soup'
        normalized_to_id['hot and sour soup'] = 12
    """
    id_to_dataset_label = {
        int(i): str(label)
        for i, label in id2label_mapping.items()
    }

    id_to_natural_label = {
        int(i): vlm_label_name(label)
        for i, label in id_to_dataset_label.items()
    }

    normalized_to_id = {
        normalize_label_text(label): int(i)
        for i, label in id_to_natural_label.items()
    }

    class_ids = sorted(id_to_dataset_label.keys())
    natural_class_names = [id_to_natural_label[i] for i in class_ids]
    dataset_class_names = [id_to_dataset_label[i] for i in class_ids]

    return {
        "class_ids": class_ids,
        "id_to_dataset_label": id_to_dataset_label,
        "id_to_natural_label": id_to_natural_label,
        "normalized_to_id": normalized_to_id,
        "natural_class_names": natural_class_names,
        "dataset_class_names": dataset_class_names,
        "num_classes": len(class_ids),
    }


def build_vlm_prompt(
    id2label_mapping,
    processor=None,
    use_chat_template=True,
):
    """
    Builds the VLM prompt.

    For SmolVLM/Idefics-style models, using the chat template is preferred
    because it inserts the correct special tokens around the image and text.
    """
    label_maps = build_vlm_label_maps(id2label_mapping)
    class_list = ", ".join(label_maps["natural_class_names"])

    user_text = (
        "You are a food image classifier.\n"
        "Choose the single best class for the image from this list:\n"
        f"{class_list}.\n\n"
        "Answer with exactly one class name from the list.\n"
        "Do not explain.\n"
        "Answer:"
    )

    if use_chat_template and processor is not None and hasattr(processor, "apply_chat_template"):
        try:
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": user_text},
                    ],
                }
            ]

            prompt = processor.apply_chat_template(
                messages,
                add_generation_prompt=True,
            )

            return prompt, label_maps

        except Exception as e:
            print("Chat template failed; falling back to manual <image> prompt.")
            print("Reason:", e)

    prompt = (
        "<image>\n"
        f"{user_text}"
    )

    return prompt, label_maps


def move_processor_inputs_to_device(inputs, device, dtype=None):
    """
    Moves processor outputs to the model device.

    Floating tensors, such as image pixel values, can be cast to fp16/bf16.
    Integer tensors, such as input_ids and attention_mask, must stay integer.
    """
    moved = {}

    for key, value in inputs.items():
        value = value.to(device)

        if dtype is not None and torch.is_floating_point(value):
            value = value.to(dtype=dtype)

        moved[key] = value

    return moved


def parse_generated_label_with_maps(generated_text, label_maps):
    """
    Parse VLM generated text into a class ID using precomputed label maps.

    Returns:
        pred_id: int, or -1 if no valid label was found
        parsed_label: natural-language label, or None
        normalized_output: normalized generated text
    """
    normalized_output = normalize_label_text(generated_text)
    normalized_to_id = label_maps["normalized_to_id"]
    id_to_natural_label = label_maps["id_to_natural_label"]

    # 1. Exact normalized match.
    if normalized_output in normalized_to_id:
        pred_id = normalized_to_id[normalized_output]
        return pred_id, id_to_natural_label[pred_id], normalized_output

    # 2. Substring match.
    # Long labels first prevents partial matching mistakes.
    candidates = sorted(
        normalized_to_id.items(),
        key=lambda x: len(x[0]),
        reverse=True,
    )

    matches = []
    for normalized_label, label_id in candidates:
        pos = normalized_output.find(normalized_label)
        if pos != -1:
            matches.append((pos, -len(normalized_label), label_id))

    if matches:
        matches.sort()
        _, _, pred_id = matches[0]
        return pred_id, id_to_natural_label[pred_id], normalized_output

    return -1, None, normalized_output

In [ ]:
# -----------------------------------------------------
# SmolVLM forced-choice class-logit scoring
# -----------------------------------------------------
# This mode makes the VLM behave like a classifier.
#
# For each image:
#   score(prompt + "bruschetta")
#   score(prompt + "caesar salad")
#   ...
#
# Returns:
#   logits: [N, num_classes]
#   labels: [N]
#
# These logits can go into the same probability-based evaluation pipeline
# used for ConvNeXt.

def compute_prompt_length(processor, image, prompt):
    """
    Computes how many tokens belong to the prompt alone.

    We use this so candidate scoring ignores the prompt tokens and scores
    only the answer tokens.
    """
    image = image.convert("RGB")

    prompt_inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt",
        padding=True,
    )

    return int(prompt_inputs["attention_mask"][0].sum().item())


def score_vlm_candidates_for_image(
    model,
    processor,
    image,
    prompt,
    prompt_len,
    candidate_names,
    device,
    dtype=None,
    candidate_batch_size=20,
    normalize_by_answer_length=True,
):
    """
    Returns one scalar score per candidate class.

    The score is the image-conditioned log-likelihood of the candidate label
    as the answer continuation.
    """
    image = image.convert("RGB")
    scores = []

    for start in range(0, len(candidate_names), candidate_batch_size):
        batch_candidates = candidate_names[start:start + candidate_batch_size]

        full_texts = [
            f"{prompt} {candidate}"
            for candidate in batch_candidates
        ]

        images = [image] * len(full_texts)

        inputs = processor(
            images=images,
            text=full_texts,
            return_tensors="pt",
            padding=True,
        )

        inputs = move_processor_inputs_to_device(
            inputs,
            device=device,
            dtype=dtype,
        )

        input_ids = inputs["input_ids"]
        attention_mask = inputs.get("attention_mask", torch.ones_like(input_ids))

        # Use the full input as target text, then mask everything except
        # the candidate answer tokens.
        labels = input_ids.clone()
        labels[:, :prompt_len] = -100
        labels[attention_mask == 0] = -100

        with torch.inference_mode():
            outputs = model(**inputs)
            logits = outputs.logits.float()

        # Causal LM scoring:
        # logits at position t predict token at position t+1.
        shift_logits = logits[:, :-1, :]
        shift_labels = labels[:, 1:]

        answer_mask = shift_labels.ne(-100)

        # Avoid invalid -100 indexing during gather.
        safe_labels = shift_labels.masked_fill(~answer_mask, 0)

        token_log_probs = F.log_softmax(shift_logits, dim=-1)

        selected_token_log_probs = token_log_probs.gather(
            dim=-1,
            index=safe_labels.unsqueeze(-1),
        ).squeeze(-1)

        token_counts = answer_mask.sum(dim=1).clamp(min=1)
        sum_log_probs = (selected_token_log_probs * answer_mask).sum(dim=1)

        if normalize_by_answer_length:
            batch_scores = sum_log_probs / token_counts
        else:
            batch_scores = sum_log_probs

        scores.extend(batch_scores.detach().cpu().numpy().tolist())

    return np.asarray(scores, dtype=np.float32)


def get_vlm_logits_and_labels(
    model,
    processor,
    dataset,
    id2label_mapping,
    device=None,
    dtype=None,
    candidate_batch_size=20,
    max_eval_samples=None,
    normalize_by_answer_length=True,
):
    """
    Converts VLM forced-choice classification into classifier-like outputs:

        logits: [num_samples, num_classes]
        labels: [num_samples]

    These can be evaluated with the same probability-based pipeline as ConvNeXt.
    """
    if device is None:
        device = next(model.parameters()).device

    if dtype is None:
        dtype = next(model.parameters()).dtype

    prompt, label_maps = build_vlm_prompt(
        id2label_mapping=id2label_mapping,
        processor=processor,
        use_chat_template=True,
    )

    candidate_names = label_maps["natural_class_names"]
    num_classes = label_maps["num_classes"]

    total = len(dataset) if max_eval_samples is None else min(len(dataset), max_eval_samples)

    logits = np.zeros((total, num_classes), dtype=np.float32)
    labels = np.zeros(total, dtype=np.int64)

    model.eval()

    # Compute prompt length once using the first image.
    # Prompt text is constant, so this avoids recomputing it for every sample.
    first_image = dataset[0]["image"]
    prompt_len = compute_prompt_length(
        processor=processor,
        image=first_image,
        prompt=prompt,
    )

    print("VLM forced-choice prompt:")
    print(prompt)
    print(f"\nPrompt length: {prompt_len} tokens")
    print(f"Scoring {total} images against {num_classes} candidate labels...")

    start_time = time.time()

    for i in tqdm(range(total), desc="Scoring VLM candidates"):
        item = dataset[i]

        image = item["image"]
        labels[i] = int(item["label"])

        logits[i] = score_vlm_candidates_for_image(
            model=model,
            processor=processor,
            image=image,
            prompt=prompt,
            prompt_len=prompt_len,
            candidate_names=candidate_names,
            device=device,
            dtype=dtype,
            candidate_batch_size=candidate_batch_size,
            normalize_by_answer_length=normalize_by_answer_length,
        )

    elapsed = time.time() - start_time

    print(f"\nForced-choice VLM scoring finished in {elapsed:.2f} seconds.")
    print(f"Average latency per image: {elapsed / max(total, 1):.2f} seconds")

    return logits, labels

## Model Evaluation and Comparison

In [ ]:
# -----------------------------
# Centralized model evaluation
# -----------------------------
# ConvNeXt and SmolVLM are both evaluated by producing logits of shape [N, 20]
# and passing those logits into the same evaluate_logits_and_plot(...) function.

# Evaluate the fine-tuned ConvNeXt-Tiny model.
print("\n--- Running ConvNeXt-Tiny Fine-tuned Evaluation ---")
convnext_logits, convnext_labels, convnext_eval_loss = get_trainer_logits_and_labels(
    trainer_convnext_hf,
    food_convnext["test"],
)

convnext_metrics = evaluate_logits_and_plot(
    logits=convnext_logits,
    labels=convnext_labels,
    id2label_mapping=id2label,
    dataset_name="Food101 20-Class Validation Set",
    num_labels=len(labels),
    model_display_name="ConvNeXt-Tiny HF Full Fine-tune",
    training_time=convnext_hf_full_train_time,
    eval_loss=convnext_eval_loss,
    normalize_cm=True,
)


#############################################################
# commented out because of memory issues and time constraints
#############################################################

# Evaluate the zero-shot SmolVLM model using forced-choice class logits.
# print("\n--- Running SmolVLM Forced-Choice Evaluation ---")

# # Set to a small integer such as 100 for a quick smoke test, or None for the full validation set.
# VLM_MAX_EVAL_SAMPLES = 20

# vlm_logits, vlm_labels = get_vlm_logits_and_labels(
#     model=model_smolvlm,
#     processor=processor_smolvlm,
#     dataset=food["test"],
#     id2label_mapping=id2label,
#     device=vlm_device,
#     dtype=vlm_dtype,
#     candidate_batch_size=2,
#     max_eval_samples=VLM_MAX_EVAL_SAMPLES,
#     normalize_by_answer_length=True,
# )

# vlm_metrics = evaluate_logits_and_plot(
#     logits=vlm_logits,
#     labels=vlm_labels,
#     id2label_mapping=id2label,
#     dataset_name="Food101 20-Class Validation Set",
#     num_labels=len(labels),
#     model_display_name="SmolVLM-Instruct Zero-Shot Forced-Choice",
#     training_time=None,
#     eval_loss=None,
#     normalize_cm=True,
# )


In [ ]:
# -----------------------------------------------------
# SmolVLM single-output generation
# -----------------------------------------------------
# This mode does NOT produce class probabilities.
#
# For each image:
#   image + prompt -> generated class name
#
# Use simple classification metrics:
#   accuracy, macro F1, weighted F1, invalid output rate, confusion matrix

def generate_single_vlm_prediction_for_image(
    model,
    processor,
    image,
    prompt,
    label_maps,
    device,
    dtype=None,
    max_new_tokens=12,
):
    """
    Runs one VLM generation call for one image.

    This is the fast/direct VLM path:
        image + prompt -> one generated class name
    """
    image = image.convert("RGB")

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt",
    )

    inputs = move_processor_inputs_to_device(
        inputs,
        device=device,
        dtype=dtype,
    )

    input_ids = inputs.get("input_ids", None)
    input_length = input_ids.shape[-1] if input_ids is not None else None

    tokenizer = getattr(processor, "tokenizer", None)
    eos_token_id = getattr(tokenizer, "eos_token_id", None) if tokenizer is not None else None
    pad_token_id = getattr(tokenizer, "pad_token_id", None) if tokenizer is not None else None

    if pad_token_id is None:
        pad_token_id = eos_token_id

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
        )

    # Decoder-only models often return prompt + generated tokens.
    # Encoder-decoder models may only return generated tokens.
    if input_length is not None and generated_ids.shape[-1] > input_length:
        new_token_ids = generated_ids[:, input_length:]
    else:
        new_token_ids = generated_ids

    raw_output = processor.batch_decode(
        new_token_ids,
        skip_special_tokens=True,
    )[0].strip()

    pred_id, parsed_label, normalized_output = parse_generated_label_with_maps(
        generated_text=raw_output,
        label_maps=label_maps,
    )

    return {
        "pred_id": pred_id,
        "parsed_label": parsed_label,
        "raw_output": raw_output,
        "normalized_output": normalized_output,
    }


def get_vlm_generated_predictions_and_labels(
    model,
    processor,
    dataset,
    id2label_mapping,
    device=None,
    dtype=None,
    max_eval_samples=None,
    max_new_tokens=12,
):
    """
    Runs direct single-label VLM generation over a dataset.

    Returns:
        predictions: [N]
            Predicted class IDs. Invalid generations are stored as -1.

        labels: [N]
            Ground-truth class IDs.

        outputs_df:
            DataFrame with raw generated text and parsed labels.

        elapsed_time:
            Total generation time in seconds.
    """
    if device is None:
        device = next(model.parameters()).device

    if dtype is None:
        dtype = next(model.parameters()).dtype

    prompt, label_maps = build_vlm_prompt(
        id2label_mapping=id2label_mapping,
        processor=processor,
        use_chat_template=True,
    )

    total = len(dataset) if max_eval_samples is None else min(len(dataset), max_eval_samples)

    predictions = np.full(total, fill_value=-1, dtype=np.int64)
    labels = np.zeros(total, dtype=np.int64)
    rows = []

    model.eval()

    print("VLM single-output generation prompt:")
    print(prompt)
    print(f"\nGenerating one class label for {total} images...")

    start_time = time.time()

    for i in tqdm(range(total), desc="Generating VLM labels"):
        item = dataset[i]

        image = item["image"]
        true_id = int(item["label"])

        result = generate_single_vlm_prediction_for_image(
            model=model,
            processor=processor,
            image=image,
            prompt=prompt,
            label_maps=label_maps,
            device=device,
            dtype=dtype,
            max_new_tokens=max_new_tokens,
        )

        pred_id = int(result["pred_id"])

        labels[i] = true_id
        predictions[i] = pred_id

        rows.append({
            "sample_index": i,
            "true_label_id": true_id,
            "true_label": label_maps["id_to_dataset_label"][true_id],
            "pred_label_id": pred_id,
            "pred_label": (
                label_maps["id_to_dataset_label"][pred_id]
                if pred_id >= 0
                else "INVALID_OUTPUT"
            ),
            "valid_output": pred_id >= 0,
            "correct": pred_id == true_id,
            "raw_output": result["raw_output"],
            "normalized_output": result["normalized_output"],
            "parsed_label": result["parsed_label"],
        })

    elapsed_time = time.time() - start_time

    outputs_df = pd.DataFrame(rows)

    print(f"\nVLM single-output generation finished in {elapsed_time:.2f} seconds.")
    print(f"Average latency per image: {elapsed_time / max(total, 1):.2f} seconds")

    return predictions, labels, outputs_df, elapsed_time

In [ ]:
# -----------------------------------------------------
# Run true single-output SmolVLM evaluation
# -----------------------------------------------------
# This is the fast/direct generation setup:
#     image + prompt -> one generated class name
#
# It does NOT produce class probabilities.
# It should only be compared using simple classification metrics:
#     accuracy, macro F1, weighted F1, invalid output rate, confusion matrix.


print("\n--- Running SmolVLM Single-Output Generation Evaluation ---")

# Start small. Increase after confirming the parsing and outputs look good.
VLM_SINGLE_OUTPUT_MAX_EVAL_SAMPLES = None

# Optional cleanup before VLM generation.
torch.cuda.empty_cache()
gc.collect()

vlm_single_predictions, vlm_single_labels, vlm_single_outputs_df, vlm_single_eval_time = (
    get_vlm_generated_predictions_and_labels(
        model=model_smolvlm,
        processor=processor_smolvlm,
        dataset=food["test"],
        id2label_mapping=id2label,
        device=vlm_device,
        dtype=vlm_dtype,
        max_eval_samples=VLM_SINGLE_OUTPUT_MAX_EVAL_SAMPLES,
        max_new_tokens=12,
    )
)

vlm_single_metrics = evaluate_predictions_and_plot(
    predictions=vlm_single_predictions,
    labels=vlm_single_labels,
    id2label_mapping=id2label,
    dataset_name="Food101 20-Class Validation Set",
    num_labels=len(labels),
    model_display_name="SmolVLM-Instruct Zero-Shot Single-Output Generation",
    outputs_df=vlm_single_outputs_df,
    normalize_cm=True,
    include_invalid_output=True,
)

In [ ]:
# -----------------------------------------------------
# Final comparison table
# -----------------------------------------------------

final_results = {
    "ConvNeXt-Tiny HF Full Fine-tune": {
        "evaluation_type": "logits",
        "accuracy": convnext_metrics["accuracy"],
        "macro_f1": convnext_metrics["macro_f1"],
        "weighted_f1": convnext_metrics["weighted_f1"],
        "top_5_accuracy": convnext_metrics["top_5_accuracy"],
        "ece": convnext_metrics["ece"],
        "nll": convnext_metrics["nll"],
        "brier_score": convnext_metrics["brier_score"],
        "invalid_output_rate": np.nan,
        "eval_time_sec": np.nan,
        "training_time_sec": convnext_hf_full_train_time,
    },
    "SmolVLM-Instruct Zero-Shot Single-Output": {
        "evaluation_type": "single-output generation",
        "accuracy": vlm_single_metrics["accuracy"],
        "macro_f1": vlm_single_metrics["macro_f1"],
        "weighted_f1": vlm_single_metrics["weighted_f1"],
        "top_5_accuracy": np.nan,
        "ece": np.nan,
        "nll": np.nan,
        "brier_score": np.nan,
        "invalid_output_rate": vlm_single_metrics["invalid_output_rate"],
        "eval_time_sec": vlm_single_eval_time,
        "training_time_sec": 0.0,
    },
}

final_comparison_df = pd.DataFrame(final_results).T

print("\n--- Final Model Comparison ---")
display(final_comparison_df.round(4))